# Module 7 — EEGCCT-3C BCI-IV-2a LOSO (22×640)

This notebook implements a compact convolutional transformer inspired by EEGCCT for
subject-independent motor-imagery EEG.

**Frozen cache interface**
- EEG shape: `(N, 22, 640)`
- Sampling rate: 160 Hz
- Classes: `left`, `right`, `feet`
- Dataset: BCI-IV-2a
- Outer evaluation: 9-subject LOSO
- No DANN, MMD, CORAL, Euclidean Alignment, or FBCSP in the first pass.

**LOSO protocol**
For each held-out target subject:
- 7 of the remaining 8 subjects are supervised training data
- 1 of the remaining 8 subjects is grouped validation data
- target subject is never used for model fitting or checkpoint selection

The final reported number is the mean test accuracy across all 9 target subjects.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / REPRODUCIBILITY / DEVICE
# ============================================================

from __future__ import annotations

import os
import gc
import json
import math
import copy
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    WeightedRandomSampler,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    cohen_kappa_score,
)

warnings.filterwarnings("ignore")

SEED = 42


def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


seed_everything(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
print("Seed:", SEED)

Device: cpu
Seed: 42


In [2]:
# ============================================================
# CELL 2 — PROJECT / CACHE DISCOVERY
# ============================================================

PROJECT_ROOT_CANDIDATES = [
    Path("/Users/ashokvarmabevara/Project2"),
    Path.cwd(),
    Path.home() / "Project2",
    Path("/mnt/data/Project2"),
]

PROJECT_ROOT = None

for root in PROJECT_ROOT_CANDIDATES:
    if (
        (root / "cross_dataset_mi_project").exists()
        or (root / "BCI IV-2a").exists()
    ):
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/Users/ashokvarmabevara/Project2")


PROJECT_DIR = (
    PROJECT_ROOT / "cross_dataset_mi_project"
)

CACHE_DIR = (
    PROJECT_DIR / "cache"
)

RESULT_DIR = (
    PROJECT_DIR
    / "results"
    / "module_7_eegcct"
)

FIG_DIR = (
    PROJECT_DIR
    / "figures"
    / "module_7_eegcct"
)

CKPT_DIR = (
    PROJECT_DIR
    / "checkpoints"
    / "module_7_eegcct"
)

for directory in [
    RESULT_DIR,
    FIG_DIR,
    CKPT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# Expected cache
# ------------------------------------------------------------

expected_cache = (
    CACHE_DIR
    / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
)


if expected_cache.exists():

    FINAL_CACHE_PATH = expected_cache

else:

    # Fallback discovery.
    h5_candidates = sorted(
        CACHE_DIR.glob("*.h5")
    )

    if not h5_candidates:
        raise FileNotFoundError(
            f"No HDF5 cache found in {CACHE_DIR}"
        )

    # Prefer filenames mentioning the frozen preprocessing.
    preferred = [
        p for p in h5_candidates
        if (
            "160hz" in p.name.lower()
            or "module_5" in p.name.lower()
        )
    ]

    FINAL_CACHE_PATH = (
        preferred[0]
        if preferred
        else h5_candidates[0]
    )


print(
    "PROJECT_ROOT:",
    PROJECT_ROOT,
)

print(
    "CACHE:",
    FINAL_CACHE_PATH,
)

assert FINAL_CACHE_PATH.exists()

PROJECT_ROOT: /Users/ashokvarmabevara/Project2
CACHE: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5


In [3]:
# ============================================================
# CELL 3 — READ CACHE METADATA
# ============================================================

def decode(v):
    if isinstance(v, bytes):
        return v.decode("utf-8")
    return str(v)


with h5py.File(
    FINAL_CACHE_PATH,
    "r",
) as h5:

    X_shape = tuple(
        h5["X"].shape
    )

    X_dtype = str(
        h5["X"].dtype
    )

    meta = {}

    for key in [
        "dataset",
        "subject",
        "run",
        "recording_id",
        "filename",
        "absolute_path",
        "harmonized_class",
    ]:

        meta[key] = [
            decode(v)
            for v in h5[
                "metadata"
            ][key][:]
        ]


cache_meta_df = pd.DataFrame(
    meta
)

cache_meta_df.insert(
    0,
    "cache_index",
    np.arange(
        len(cache_meta_df),
        dtype=np.int64,
    ),
)

assert X_shape[1:] == (
    22,
    640,
)

assert X_dtype == "float32"


CLASSES = [
    "left",
    "right",
    "feet",
]

CLASS_TO_ID = {
    c: i
    for i, c in enumerate(CLASSES)
}

ID_TO_CLASS = {
    i: c
    for c, i in CLASS_TO_ID.items()
}

N_CLASSES = 3

assert set(
    cache_meta_df[
        "harmonized_class"
    ].unique()
).issubset(
    set(CLASSES)
)


bci_meta = cache_meta_df[
    cache_meta_df[
        "dataset"
    ].astype(str)
    == "BCI-IV-2a"
].copy()

bci_meta["subject"] = (
    bci_meta["subject"]
    .astype(str)
)


BCI_SUBJECTS = sorted(
    bci_meta[
        "subject"
    ].unique()
)

assert len(
    BCI_SUBJECTS
) == 9


print(
    "Cache shape:",
    X_shape,
)

print(
    "BCI subjects:",
    BCI_SUBJECTS,
)

print(
    "BCI epochs:",
    len(bci_meta),
)

display(
    bci_meta.groupby(
        "subject"
    )[
        "harmonized_class"
    ]
    .value_counts()
    .unstack(
        fill_value=0
    )
    .reindex(
        index=BCI_SUBJECTS,
        columns=CLASSES,
        fill_value=0,
    )
)

Cache shape: (9316, 22, 640)
BCI subjects: ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09']
BCI epochs: 1944


harmonized_class,left,right,feet
subject,,,
S01,72,72,72
S02,72,72,72
S03,72,72,72
S04,72,72,72
S05,72,72,72
S06,72,72,72
S07,72,72,72
S08,72,72,72
S09,72,72,72


In [4]:
# ============================================================
# CELL 4 — HDF5 ACCESS + NUMERICAL QA
# ============================================================

def load_indices(
    indices,
):
    """
    Load selected epochs from the frozen HDF5 cache.
    """

    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    with h5py.File(
        FINAL_CACHE_PATH,
        "r",
    ) as h5:

        X = np.asarray(
            h5["X"][indices],
            dtype=np.float32,
        )

    return X


qa_indices = np.arange(
    min(
        512,
        X_shape[0],
    ),
    dtype=np.int64,
)

X_qa = load_indices(
    qa_indices
)

nonfinite = int(
    (
        ~np.isfinite(
            X_qa
        )
    ).sum()
)

zero_var = int(
    (
        np.var(
            X_qa,
            axis=(1, 2),
        )
        <= 1e-12
    ).sum()
)

print(
    "QA shape:",
    X_qa.shape,
)

print(
    "Non-finite values:",
    nonfinite,
)

print(
    "Zero-variance epochs:",
    zero_var,
)

assert nonfinite == 0

print(
    "✅ Cache numerical QA passed."
)

QA shape: (512, 22, 640)
Non-finite values: 0
Zero-variance epochs: 0
✅ Cache numerical QA passed.


In [5]:
# ============================================================
# CELL 5 — SOURCE-ONLY ROBUST NORMALIZATION
# ============================================================

class SourceRobustNormalizer:
    """
    Channel-wise robust scaling.

    IMPORTANT:
    fitted only on the current outer-fold source data.
    """

    def __init__(
        self,
        eps=1e-6,
    ):

        self.eps = eps
        self.median_ = None
        self.iqr_ = None

    def fit(
        self,
        X,
    ):

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        values = (
            X
            .transpose(
                1,
                0,
                2,
            )
            .reshape(
                X.shape[1],
                -1,
            )
        )

        self.median_ = np.median(
            values,
            axis=1,
        )

        q25 = np.percentile(
            values,
            25,
            axis=1,
        )

        q75 = np.percentile(
            values,
            75,
            axis=1,
        )

        self.iqr_ = np.maximum(
            q75 - q25,
            self.eps,
        )

        return self

    def transform(
        self,
        X,
    ):

        if self.median_ is None:
            raise RuntimeError(
                "Normalizer is not fitted."
            )

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        Z = (
            X
            - self.median_[
                None,
                :,
                None,
            ]
        )

        Z = Z / (
            self.iqr_[
                None,
                :,
                None,
            ]
            + self.eps
        )

        Z = np.nan_to_num(
            Z,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        return Z.astype(
            np.float32
        )


print(
    "✅ Source-only robust normalizer ready."
)

✅ Source-only robust normalizer ready.


In [6]:
# ============================================================
# CELL 6 — PAPER-STYLE GROUPED LOSO SPLIT
# ============================================================

def choose_train_val_subjects(
    source_subjects,
    seed,
):
    """
    Select 1 validation subject and use all other
    source subjects for supervised training.

    With one outer target:
        8 source subjects
        7 train
        1 validation
        1 target test
    """

    source_subjects = sorted(
        [
            str(s)
            for s in source_subjects
        ]
    )

    rng = np.random.default_rng(
        seed
    )

    shuffled = (
        source_subjects.copy()
    )

    rng.shuffle(
        shuffled
    )

    val_subject = (
        str(
            shuffled[0]
        )
    )

    train_subjects = sorted(
        shuffled[1:]
    )

    assert len(
        train_subjects
    ) == 7

    assert val_subject not in (
        train_subjects
    )

    return (
        train_subjects,
        val_subject,
    )


print(
    "✅ Grouped LOSO split ready."
)

✅ Grouped LOSO split ready.


In [7]:
# ============================================================
# CELL 7 — EEGCCT CONVOLUTIONAL TOKENIZER
# ============================================================

class EEGCCTConvTokenizer(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        temporal_filters=32,
        token_dim=64,
        temporal_kernel=25,
        pool_size=4,
        token_kernel=8,
        token_stride=4,
    ):

        super().__init__()

        self.temporal = nn.Sequential(

            nn.Conv2d(
                1,
                temporal_filters,
                kernel_size=(
                    1,
                    temporal_kernel,
                ),
                padding=(
                    0,
                    temporal_kernel // 2,
                ),
                bias=False,
            ),

            nn.BatchNorm2d(
                temporal_filters
            ),

            nn.ELU(),
        )

        # Depthwise spatial filtering across all EEG channels.
        self.spatial = nn.Sequential(

            nn.Conv2d(
                temporal_filters,
                temporal_filters,
                kernel_size=(
                    n_channels,
                    1,
                ),
                groups=temporal_filters,
                bias=False,
            ),

            nn.BatchNorm2d(
                temporal_filters
            ),

            nn.ELU(),
        )

        self.pool = nn.AvgPool2d(
            kernel_size=(
                1,
                pool_size,
            ),
            stride=(
                1,
                pool_size,
            ),
        )

        self.dropout = nn.Dropout(
            0.25
        )

        # Convolutional tokenizer:
        # local temporal features -> tokens.
        self.tokenizer = nn.Conv1d(
            temporal_filters,
            token_dim,
            kernel_size=token_kernel,
            stride=token_stride,
            padding=token_kernel // 2,
            bias=True,
        )

        self.token_norm = nn.BatchNorm1d(
            token_dim
        )

    def forward(
        self,
        x,
    ):
        # x = (B,C,T)

        x = x.unsqueeze(
            1
        )

        z = self.temporal(
            x
        )

        z = self.spatial(
            z
        )

        z = self.pool(
            z
        )

        z = self.dropout(
            z
        )

        # B,F,1,T -> B,F,T
        z = z.squeeze(
            2
        )

        z = self.tokenizer(
            z
        )

        z = self.token_norm(
            z
        )

        z = F.gelu(
            z
        )

        # B,D,L -> B,L,D
        z = z.transpose(
            1,
            2,
        )

        return z


print(
    "✅ EEGCCT tokenizer ready."
)

✅ EEGCCT tokenizer ready.


In [8]:
# ============================================================
# CELL 8 — EEGCCT TRANSFORMER BLOCK
# ============================================================

class TransformerEncoderBlock(
    nn.Module
):

    def __init__(
        self,
        dim=64,
        heads=4,
        mlp_ratio=2.0,
        dropout=0.20,
    ):

        super().__init__()

        self.norm1 = nn.LayerNorm(
            dim
        )

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )

        self.drop_path1 = nn.Dropout(
            dropout
        )

        self.norm2 = nn.LayerNorm(
            dim
        )

        hidden = int(
            dim * mlp_ratio
        )

        self.mlp = nn.Sequential(

            nn.Linear(
                dim,
                hidden,
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden,
                dim,
            ),

            nn.Dropout(
                dropout
            ),
        )

    def forward(
        self,
        x,
    ):

        h = self.norm1(
            x
        )

        attn, _ = self.attn(
            h,
            h,
            h,
            need_weights=False,
        )

        x = (
            x
            + self.drop_path1(
                attn
            )
        )

        x = (
            x
            + self.mlp(
                self.norm2(
                    x
                )
            )
        )

        return x


class SequencePooling(
    nn.Module
):

    def __init__(
        self,
        dim=64,
    ):

        super().__init__()

        self.attention = nn.Linear(
            dim,
            1,
        )

    def forward(
        self,
        x,
    ):

        # x = (B,L,D)

        scores = self.attention(
            x
        )

        weights = torch.softmax(
            scores,
            dim=1,
        )

        pooled = (
            x
            * weights
        ).sum(
            dim=1
        )

        return pooled


print(
    "✅ Transformer and sequence pooling ready."
)

✅ Transformer and sequence pooling ready.


In [9]:
# ============================================================
# CELL 9 — EEGCCT-3C MODEL
# ============================================================

class EEGCCT3C(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        n_samples=640,
        n_classes=3,
        temporal_filters=32,
        token_dim=64,
        depth=3,
        heads=4,
        dropout=0.20,
    ):

        super().__init__()

        self.tokenizer = (
            EEGCCTConvTokenizer(
                n_channels=n_channels,
                temporal_filters=temporal_filters,
                token_dim=token_dim,
                temporal_kernel=25,
                pool_size=4,
                token_kernel=8,
                token_stride=4,
            )
        )

        # Estimate token count dynamically.
        with torch.no_grad():

            dummy = torch.zeros(
                2,
                n_channels,
                n_samples,
            )

            tokens = (
                self.tokenizer(
                    dummy
                )
            )

            self.n_tokens = (
                tokens.shape[1]
            )

        self.position = nn.Parameter(
            torch.zeros(
                1,
                self.n_tokens,
                token_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position,
            std=0.02,
        )

        self.transformer = (
            nn.ModuleList(
                [
                    TransformerEncoderBlock(
                        dim=token_dim,
                        heads=heads,
                        mlp_ratio=2.0,
                        dropout=dropout,
                    )
                    for _ in range(
                        depth
                    )
                ]
            )
        )

        self.norm = nn.LayerNorm(
            token_dim
        )

        self.pool = SequencePooling(
            token_dim
        )

        self.head = nn.Sequential(

            nn.Linear(
                token_dim,
                64,
            ),

            nn.GELU(),

            nn.Dropout(
                0.35
            ),

            nn.Linear(
                64,
                n_classes,
            ),
        )

    def forward(
        self,
        x,
    ):

        z = self.tokenizer(
            x
        )

        z = (
            z
            + self.position[
                :,
                :z.shape[1],
                :,
            ]
        )

        for block in (
            self.transformer
        ):

            z = block(
                z
            )

        z = self.norm(
            z
        )

        pooled = self.pool(
            z
        )

        logits = self.head(
            pooled
        )

        return logits


# ------------------------------------------------------------
# Forward test
# ------------------------------------------------------------

_test_model = EEGCCT3C().to(
    device
)

with torch.no_grad():

    test_x = torch.randn(
        4,
        22,
        640,
        device=device,
    )

    test_logits = _test_model(
        test_x
    )

parameter_count = sum(
    p.numel()
    for p in _test_model.parameters()
    if p.requires_grad
)

print(
    "Token count:",
    _test_model.n_tokens,
)

print(
    "Parameters:",
    f"{parameter_count:,}",
)

print(
    "Output:",
    tuple(
        test_logits.shape
    ),
)

assert tuple(
    test_logits.shape
) == (
    4,
    3,
)

print(
    "✅ EEGCCT-3C forward PASS."
)

Token count: 41
Parameters: 125,796
Output: (4, 3)
✅ EEGCCT-3C forward PASS.


In [10]:
# ============================================================
# CELL 10 — EEG AUGMENTATION + BALANCED LOADER
# ============================================================

def augment_eegcct(
    x,
):
    """
    Conservative MI-EEG augmentation.
    """

    x = x.clone()

    B, C, T = x.shape

    # Amplitude scaling.
    if torch.rand(
        1,
        device=x.device,
    ).item() < 0.35:

        scale = torch.empty(
            B,
            1,
            1,
            device=x.device,
        ).uniform_(
            0.90,
            1.10,
        )

        x = (
            x
            * scale
        )

    # Low-amplitude Gaussian noise.
    if torch.rand(
        1,
        device=x.device,
    ).item() < 0.25:

        x = (
            x
            + 0.004
            * torch.randn_like(
                x
            )
        )

    # Small temporal shift.
    if torch.rand(
        1,
        device=x.device,
    ).item() < 0.20:

        max_shift = 8

        shifts = torch.randint(
            -max_shift,
            max_shift + 1,
            (
                B,
            ),
            device=x.device,
        )

        x = torch.stack(
            [
                torch.roll(
                    x[i],
                    int(
                        shifts[i].item()
                    ),
                    dims=-1,
                )
                for i in range(B)
            ],
            dim=0,
        )

    return x


def make_train_loader(
    X,
    y,
    batch_size=64,
):

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    y = np.asarray(
        y,
        dtype=np.int64,
    )

    dataset = TensorDataset(
        torch.from_numpy(
            X
        ),
        torch.from_numpy(
            y
        ),
    )

    counts = np.bincount(
        y,
        minlength=N_CLASSES,
    ).astype(
        np.float64
    )

    class_weights = np.zeros(
        N_CLASSES,
        dtype=np.float64,
    )

    valid = (
        counts > 0
    )

    class_weights[
        valid
    ] = (
        1.0
        / counts[
            valid
        ]
    )

    sample_weights = (
        class_weights[
            y
        ]
    )

    sampler = (
        WeightedRandomSampler(
            torch.as_tensor(
                sample_weights,
                dtype=torch.double,
            ),
            num_samples=len(y),
            replacement=True,
        )
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )


def make_eval_loader(
    X,
    batch_size=128,
):

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    dataset = TensorDataset(
        torch.from_numpy(
            X
        ),
        torch.zeros(
            len(X),
            dtype=torch.long,
        ),
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )


print(
    "✅ EEGCCT data pipeline ready."
)

✅ EEGCCT data pipeline ready.


In [11]:
# ============================================================
# CELL 11 — TRAIN / PREDICT EEGCCT
# ============================================================

@torch.no_grad()
def predict_eegcct(
    model,
    X,
):
    """
    Returns probabilities.
    """

    model.eval()

    loader = make_eval_loader(
        X
    )

    logits_all = []

    for xb, _ in loader:

        xb = xb.to(
            device,
            non_blocking=True,
        )

        logits = model(
            xb
        )

        logits_all.append(
            logits.detach()
            .cpu()
            .numpy()
        )

    logits = np.concatenate(
        logits_all,
        axis=0,
    )

    logits = (
        logits
        - logits.max(
            axis=1,
            keepdims=True,
        )
    )

    probs = np.exp(
        logits
    )

    probs /= (
        probs.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return probs.astype(
        np.float32
    )


def train_eegcct(
    X_train,
    y_train,
    X_val,
    y_val,
    seed=42,
    epochs=150,
    batch_size=64,
    lr=7e-4,
    patience=25,
):

    seed_everything(
        seed
    )

    model = EEGCCT3C(
        n_channels=22,
        n_samples=640,
        n_classes=3,
        temporal_filters=32,
        token_dim=64,
        depth=3,
        heads=4,
        dropout=0.20,
    ).to(device)

    criterion = nn.CrossEntropyLoss(
        weight=(
            lambda counts: torch.tensor(
                (
                    counts.sum()
                    /
                    (
                        3.0
                        * np.maximum(
                            counts,
                            1.0,
                        )
                    )
                )
                /
                (
                    (
                        counts.sum()
                        /
                        (
                            3.0
                            * np.maximum(
                                counts,
                                1.0,
                            )
                        )
                    ).mean()
                    + 1e-12
                ),
                dtype=torch.float32,
                device=device,
            )
        )(
            np.bincount(
                y_train,
                minlength=3,
            ).astype(
                np.float32
            )
        ),
        label_smoothing=0.01,
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=3e-4,
        betas=(
            0.9,
            0.999,
        ),
    )

    scheduler = (
        optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=20,
            T_mult=2,
            eta_min=2e-6,
        )
    )

    loader = make_train_loader(
        X_train,
        y_train,
        batch_size=batch_size,
    )

    best_state = None
    best_loss = np.inf
    best_bacc = -np.inf
    best_epoch = 0
    wait = 0

    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        epoch_loss = []

        train_true = []
        train_pred = []

        for step, (
            xb,
            yb,
        ) in enumerate(
            loader
        ):

            xb = xb.to(
                device,
                non_blocking=True,
            )

            yb = yb.to(
                device,
                non_blocking=True,
            )

            xb = augment_eegcct(
                xb
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                xb
            )

            loss = criterion(
                logits,
                yb,
            )

            if not torch.isfinite(
                loss
            ):
                continue

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=2.0,
            )

            optimizer.step()

            scheduler.step(
                (
                    epoch - 1
                )
                + (
                    step
                    / max(
                        1,
                        len(
                            loader
                        ),
                    )
                )
            )

            epoch_loss.append(
                float(
                    loss.item()
                )
            )

            train_true.extend(
                yb.detach()
                .cpu()
                .numpy()
            )

            train_pred.extend(
                logits.argmax(
                    dim=1
                )
                .detach()
                .cpu()
                .numpy()
            )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        P_val = predict_eegcct(
            model,
            X_val,
        )

        pred_val = (
            P_val.argmax(
                axis=1
            )
        )

        val_loss = -float(
            np.mean(
                np.log(
                    np.clip(
                        P_val[
                            np.arange(
                                len(y_val)
                            ),
                            y_val,
                        ],
                        1e-8,
                        1.0,
                    )
                )
            )
        )

        train_acc = (
            accuracy_score(
                train_true,
                train_pred,
            )
            * 100.0
        )

        val_acc = (
            accuracy_score(
                y_val,
                pred_val,
            )
            * 100.0
        )

        val_bacc = (
            balanced_accuracy_score(
                y_val,
                pred_val,
            )
            * 100.0
        )

        history.append(
            {
                "epoch":
                    epoch,
                "loss":
                    float(
                        np.mean(
                            epoch_loss
                        )
                    ),
                "val_loss":
                    val_loss,
                "train_acc":
                    train_acc,
                "val_acc":
                    val_acc,
                "val_bacc":
                    val_bacc,
                "lr":
                    optimizer.param_groups[
                        0
                    ]["lr"],
            }
        )

        # ----------------------------------------------------
        # Validation-loss checkpoint
        # ----------------------------------------------------

        if val_loss < (
            best_loss
            - 1e-5
        ):

            best_loss = val_loss
            best_bacc = val_bacc
            best_epoch = epoch
            wait = 0

            best_state = copy.deepcopy(
                model.state_dict()
            )

        else:

            wait += 1

        if (
            epoch == 1
            or epoch % 10 == 0
        ):

            print(
                f"    epoch {epoch:03d} | "
                f"train={train_acc:5.1f}% | "
                f"val={val_acc:5.1f}% | "
                f"bAcc={val_bacc:5.1f}% | "
                f"vLoss={val_loss:.4f} | "
                f"lr={optimizer.param_groups[0]['lr']:.2e}"
            )

        if wait >= patience:

            print(
                f"    early stop at "
                f"epoch {epoch}; "
                f"best={best_epoch}"
            )

            break

    if best_state is None:
        raise RuntimeError(
            "No valid EEGCCT checkpoint."
        )

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
        best_epoch,
        best_loss,
        best_bacc,
    )


print(
    "✅ EEGCCT training functions ready."
)

✅ EEGCCT training functions ready.


In [12]:
# ============================================================
# CELL 12 — ONE EEGCCT LOSO FOLD
# ============================================================

def run_eegcct_fold(
    target_subject,
    fold_id,
    seed=SEED,
    epochs=150,
    batch_size=64,
    lr=7e-4,
    patience=25,
    n_seeds=2,
):
    """
    Outer LOSO fold.

    1 target subject is fully held out.
    1 of the remaining 8 is grouped validation.
    7 remaining subjects are training.
    """

    t0 = time.time()

    target_subject = str(
        target_subject
    )

    source_subjects = [
        s
        for s in BCI_SUBJECTS
        if s != target_subject
    ]

    train_subjects, val_subject = (
        choose_train_val_subjects(
            source_subjects,
            seed=seed,
        )
    )

    # --------------------------------------------------------
    # Metadata masks
    # --------------------------------------------------------

    source_mask = (
        bci_meta[
            "subject"
        ].isin(
            source_subjects
        )
    )

    target_mask = (
        bci_meta[
            "subject"
        ]
        == target_subject
    )

    train_mask = (
        bci_meta[
            "subject"
        ].isin(
            train_subjects
        )
    )

    val_mask = (
        bci_meta[
            "subject"
        ]
        == val_subject
    )

    train_indices = (
        bci_meta.loc[
            train_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    val_indices = (
        bci_meta.loc[
            val_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    target_indices = (
        bci_meta.loc[
            target_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    # --------------------------------------------------------
    # Load
    # --------------------------------------------------------

    X_train_raw = load_indices(
        train_indices
    )

    X_val_raw = load_indices(
        val_indices
    )

    X_test_raw = load_indices(
        target_indices
    )

    y_train = (
        bci_meta.loc[
            train_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    y_val = (
        bci_meta.loc[
            val_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    y_test = (
        bci_meta.loc[
            target_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    # --------------------------------------------------------
    # Source-only normalizer
    # --------------------------------------------------------

    normalizer = (
        SourceRobustNormalizer()
        .fit(
            X_train_raw
        )
    )

    X_train = (
        normalizer.transform(
            X_train_raw
        )
    )

    X_val = (
        normalizer.transform(
            X_val_raw
        )
    )

    X_test = (
        normalizer.transform(
            X_test_raw
        )
    )

    # --------------------------------------------------------
    # Report
    # --------------------------------------------------------

    print(
        "\n"
        + "=" * 78
    )

    print(
        f"EEGCCT-3C LOSO "
        f"[{fold_id}/9] — target {target_subject}"
    )

    print(
        "=" * 78
    )

    print(
        "Train subjects:",
        train_subjects,
    )

    print(
        "Validation subject:",
        val_subject,
    )

    print(
        "Target subject:",
        target_subject,
    )

    print(
        "Train trials:",
        len(X_train),
    )

    print(
        "Val trials:",
        len(X_val),
    )

    print(
        "Test trials:",
        len(X_test),
    )

    # --------------------------------------------------------
    # Train multiple seeds
    # --------------------------------------------------------

    models = []
    histories = []
    seed_rows = []

    for k in range(
        n_seeds
    ):

        model_seed = (
            seed
            + 1009 * k
        )

        (
            model,
            history,
            best_epoch,
            best_loss,
            best_bacc,
        ) = train_eegcct(
            X_train,
            y_train,
            X_val,
            y_val,
            seed=model_seed,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            patience=patience,
        )

        models.append(
            model
        )

        histories.append(
            history
        )

        seed_rows.append(
            {
                "seed":
                    model_seed,
                "best_epoch":
                    best_epoch,
                "best_val_loss":
                    best_loss,
                "best_val_bacc":
                    best_bacc,
            }
        )

    seed_summary = pd.DataFrame(
        seed_rows
    )

    display(
        seed_summary
    )

    # --------------------------------------------------------
    # Validation ensemble
    # --------------------------------------------------------

    val_probs = [
        predict_eegcct(
            model,
            X_val,
        )
        for model in models
    ]

    P_val = np.mean(
        np.stack(
            val_probs,
            axis=0,
        ),
        axis=0,
    )

    val_pred = (
        P_val.argmax(
            axis=1
        )
    )

    val_acc = (
        accuracy_score(
            y_val,
            val_pred,
        )
        * 100.0
    )

    val_bacc = (
        balanced_accuracy_score(
            y_val,
            val_pred,
        )
        * 100.0
    )

    # --------------------------------------------------------
    # External test ensemble
    # --------------------------------------------------------

    test_probs = [
        predict_eegcct(
            model,
            X_test,
        )
        for model in models
    ]

    P_test = np.mean(
        np.stack(
            test_probs,
            axis=0,
        ),
        axis=0,
    )

    test_pred = (
        P_test.argmax(
            axis=1
        )
    )

    test_acc = (
        accuracy_score(
            y_test,
            test_pred,
        )
        * 100.0
    )

    test_bacc = (
        balanced_accuracy_score(
            y_test,
            test_pred,
        )
        * 100.0
    )

    kappa = (
        cohen_kappa_score(
            y_test,
            test_pred,
        )
    )

    elapsed = (
        time.time()
        - t0
    )

    print(
        "\n"
        + "-" * 78
    )

    print(
        f"Target subject       : {target_subject}"
    )

    print(
        f"Validation ensemble  : {val_bacc:.2f}%"
    )

    print(
        f"External test acc    : {test_acc:.2f}%"
    )

    print(
        f"External test bAcc   : {test_bacc:.2f}%"
    )

    print(
        f"Kappa                : {kappa:.4f}"
    )

    print(
        f"Elapsed              : "
        f"{elapsed / 60.0:.1f} min"
    )

    print(
        "-" * 78
    )

    return {
        "fold":
            fold_id,

        "subject":
            target_subject,

        "train_subjects":
            train_subjects,

        "val_subject":
            val_subject,

        "test_acc":
            float(
                test_acc
            ),

        "test_bacc":
            float(
                test_bacc
            ),

        "val_acc":
            float(
                val_acc
            ),

        "val_bacc":
            float(
                val_bacc
            ),

        "kappa":
            float(
                kappa
            ),

        "y_test":
            y_test,

        "pred_test":
            test_pred,

        "P_test":
            P_test,

        "models":
            models,

        "histories":
            histories,

        "seed_summary":
            seed_summary,
    }


print(
    "✅ LOSO fold function ready."
)

✅ LOSO fold function ready.


In [13]:
# ============================================================
# CELL 13 — S01 SMOKE TEST
# ============================================================

RUN_SMOKE = True

if RUN_SMOKE:

    smoke_result = run_eegcct_fold(
        target_subject="S01",
        fold_id=1,
        seed=SEED,
        epochs=150,
        batch_size=64,
        lr=7e-4,
        patience=25,
        n_seeds=2,
    )

    print(
        "\n"
        + "=" * 78
    )

    print(
        "EEGCCT-3C S01 SMOKE TEST"
    )

    print(
        "=" * 78
    )

    print(
        f"S01 accuracy : "
        f"{smoke_result['test_acc']:.2f}%"
    )

    print(
        f"S01 bAcc     : "
        f"{smoke_result['test_bacc']:.2f}%"
    )


EEGCCT-3C LOSO [1/9] — target S01
Train subjects: ['S02', 'S03', 'S04', 'S06', 'S07', 'S08', 'S09']
Validation subject: S05
Target subject: S01
Train trials: 1512
Val trials: 216
Test trials: 216
    epoch 001 | train= 37.8% | val= 33.3% | bAcc= 33.3% | vLoss=1.1076 | lr=6.96e-04
    epoch 010 | train= 63.0% | val= 32.9% | bAcc= 32.9% | vLoss=1.1135 | lr=3.53e-04
    epoch 020 | train= 72.4% | val= 35.2% | bAcc= 35.2% | vLoss=1.3185 | lr=2.01e-06
    early stop at epoch 26; best=1
    epoch 001 | train= 35.8% | val= 33.8% | bAcc= 33.8% | vLoss=1.0972 | lr=6.96e-04
    epoch 010 | train= 65.7% | val= 32.9% | bAcc= 32.9% | vLoss=1.2346 | lr=3.53e-04
    epoch 020 | train= 71.4% | val= 32.9% | bAcc= 32.9% | vLoss=1.3518 | lr=2.01e-06
    early stop at epoch 26; best=1


,seed,best_epoch,best_val_loss,best_val_bacc
0,42,1,1.107571,33.333333
1,1051,1,1.097212,33.796296



------------------------------------------------------------------------------
Target subject       : S01
Validation ensemble  : 33.33%
External test acc    : 51.39%
External test bAcc   : 51.39%
Kappa                : 0.2708
Elapsed              : 3.7 min
------------------------------------------------------------------------------

EEGCCT-3C S01 SMOKE TEST
S01 accuracy : 51.39%
S01 bAcc     : 51.39%


In [ ]:
# ============================================================
# CELL 14 — FULL 9-SUBJECT EEGCCT LOSO
# ============================================================

RUN_FULL_LOSO = False

if RUN_FULL_LOSO:

    final_results = []
    fold_objects = {}

    for fold_id, subject in enumerate(
        BCI_SUBJECTS,
        start=1,
    ):

        result = run_eegcct_fold(
            target_subject=subject,
            fold_id=fold_id,
            seed=SEED,
            epochs=150,
            batch_size=64,
            lr=7e-4,
            patience=25,
            n_seeds=2,
        )

        fold_objects[
            subject
        ] = result

        final_results.append(
            {
                "fold":
                    fold_id,

                "subject":
                    subject,

                "val_subject":
                    result[
                        "val_subject"
                    ],

                "val_bacc":
                    result[
                        "val_bacc"
                    ],

                "test_accuracy":
                    result[
                        "test_acc"
                    ],

                "test_bacc":
                    result[
                        "test_bacc"
                    ],

                "kappa":
                    result[
                        "kappa"
                    ],
            }
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    eegcct_results_df = (
        pd.DataFrame(
            final_results
        )
    )

    print(
        "\n"
        + "=" * 78
    )

    print(
        "EEGCCT-3C — BCI-IV-2a LOSO SUMMARY"
    )

    print(
        "=" * 78
    )

    display(
        eegcct_results_df
    )

    mean_acc = (
        eegcct_results_df[
            "test_accuracy"
        ].mean()
    )

    mean_bacc = (
        eegcct_results_df[
            "test_bacc"
        ].mean()
    )

    median_acc = (
        eegcct_results_df[
            "test_accuracy"
        ].median()
    )

    std_acc = (
        eegcct_results_df[
            "test_accuracy"
        ].std()
    )

    n70 = int(
        (
            eegcct_results_df[
                "test_accuracy"
            ]
            >= 70.0
        ).sum()
    )

    n80 = int(
        (
            eegcct_results_df[
                "test_accuracy"
            ]
            >= 80.0
        ).sum()
    )

    print(
        f"\nMean accuracy : {mean_acc:.2f}%"
    )

    print(
        f"Mean bAcc     : {mean_bacc:.2f}%"
    )

    print(
        f"Median        : {median_acc:.2f}%"
    )

    print(
        f"Std           : {std_acc:.2f}%"
    )

    print(
        f"Subjects >=70 : {n70}/9"
    )

    print(
        f"Subjects >=80 : {n80}/9"
    )

    if mean_acc >= 80.0:

        print(
            "\n✅ 80% TARGET ACHIEVED."
        )

    elif mean_acc >= 70.0:

        print(
            "\n✅ 70% TARGET ACHIEVED."
        )

    else:

        print(
            "\n❌ Target not yet achieved."
        )

In [ ]:
# ============================================================
# CELL 15 — CONFUSION MATRIX / REPORT / SAVE
# ============================================================

if (
    "fold_objects" in globals()
    and len(fold_objects) > 0
):

    y_all = np.concatenate(
        [
            fold_objects[s][
                "y_test"
            ]
            for s in BCI_SUBJECTS
        ]
    )

    p_all = np.concatenate(
        [
            fold_objects[s][
                "pred_test"
            ]
            for s in BCI_SUBJECTS
        ]
    )

    cm = confusion_matrix(
        y_all,
        p_all,
        labels=list(
            range(
                N_CLASSES
            )
        ),
        normalize="true",
    )

    print(
        "Normalized confusion matrix:"
    )

    display(
        pd.DataFrame(
            cm,
            index=CLASSES,
            columns=CLASSES,
        ).round(3)
    )

    print(
        "\nClassification report:"
    )

    print(
        classification_report(
            y_all,
            p_all,
            labels=list(
                range(
                    N_CLASSES
                )
            ),
            target_names=CLASSES,
            digits=4,
        )
    )

    plt.figure(
        figsize=(6, 5)
    )

    plt.imshow(
        cm,
        interpolation="nearest",
    )

    plt.xticks(
        range(
            N_CLASSES
        ),
        CLASSES,
    )

    plt.yticks(
        range(
            N_CLASSES
        ),
        CLASSES,
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "True"
    )

    plt.title(
        "EEGCCT-3C — BCI-IV-2a LOSO"
    )

    for i in range(
        N_CLASSES
    ):

        for j in range(
            N_CLASSES
        ):

            plt.text(
                j,
                i,
                f"{cm[i,j]:.2f}",
                ha="center",
                va="center",
            )

    plt.tight_layout()

    cm_path = (
        FIG_DIR
        / "eegcct_3c_loso_confusion_matrix.png"
    )

    plt.savefig(
        cm_path,
        dpi=180,
    )

    plt.show()

    # --------------------------------------------------------
    # Save result table
    # --------------------------------------------------------

    csv_path = (
        RESULT_DIR
        / "eegcct_3c_loso_results.csv"
    )

    eegcct_results_df.to_csv(
        csv_path,
        index=False,
    )

    print(
        "Saved:",
        csv_path,
    )


print(
    "✅ Reporting cell ready."
)

## Evaluation note

The published EEGCCT work reports a 70.12% mean accuracy for EEGCCT-2 under its BCI Competition evaluation and describes LOSO subject-independent evaluation with 7 training subjects, 1 validation subject, and 1 held-out test subject. That published result uses the original BCI-IV-2a setup; this notebook adapts the architecture to your frozen `(22, 640)` three-class cache, so the published percentage is a benchmark rather than a guaranteed outcome. citeturn560140search1turn758041search0